# Advanced Retrieval with LangChain

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

In [1]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [2]:
os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

In [3]:
# BREAKOUT ROOM PART #2: Complete Setup and Dependencies
# ======================================================

print("🔧 Setting up ALL dependencies for Breakout Room Part #2...")

# 1. LangSmith Setup (FIRST - before any other operations)
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("Enter your LangSmith API Key:")
from uuid import uuid4
os.environ["LANGCHAIN_PROJECT"] = f"Advanced_Retrieval_Evaluation_{uuid4().hex[0:8]}"

# 2. NLTK Setup (required for Ragas)
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

# 3. Ragas Imports
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.testset import TestsetGenerator
from ragas import evaluate, EvaluationDataset
from ragas.metrics import (
    LLMContextRecall, 
    Faithfulness, 
    FactualCorrectness, 
    ResponseRelevancy, 
    ContextEntityRecall, 
    NoiseSensitivity,
    ContextPrecision
    )
from ragas import RunConfig

# 4. LangSmith Client
from langsmith import Client
client = Client()

# 5. Test LangSmith connection
from langchain_openai import ChatOpenAI
test_llm = ChatOpenAI(model="gpt-4.1-nano")
test_response = test_llm.invoke("Test message for LangSmith tracing")

print("✅ ALL dependencies loaded successfully!")
print("✅ LangSmith environment configured!")
print("✅ NLTK packages downloaded!")
print("✅ Ragas components imported!")
print("✅ LangSmith tracing working!")

🔧 Setting up ALL dependencies for Breakout Room Part #2...


[nltk_data] Downloading package punkt to /home/saimo/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/saimo/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


✅ ALL dependencies loaded successfully!
✅ LangSmith environment configured!
✅ NLTK packages downloaded!
✅ Ragas components imported!
✅ LangSmith tracing working!


## Task 2: Data Collection and Preparation

We'll be using our Loan Data once again - this time the strutured data available through the CSV!

### Data Preparation

We want to make sure all our documents have the relevant metadata for the various retrieval strategies we're going to be applying today.

In [4]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from datetime import datetime, timedelta

loader = CSVLoader(
    file_path=f"./data/complaints.csv",
    metadata_columns=[
      "Date received", 
      "Product", 
      "Sub-product", 
      "Issue", 
      "Sub-issue", 
      "Consumer complaint narrative", 
      "Company public response", 
      "Company", 
      "State", 
      "ZIP code", 
      "Tags", 
      "Consumer consent provided?", 
      "Submitted via", 
      "Date sent to company", 
      "Company response to consumer", 
      "Timely response?", 
      "Consumer disputed?", 
      "Complaint ID"
    ]
)

loan_complaint_data = loader.load()

for doc in loan_complaint_data:
    doc.page_content = doc.metadata["Consumer complaint narrative"]

Let's look at an example document to see if everything worked as expected!

In [5]:
loan_complaint_data[0]

Document(metadata={'source': './data/complaints.csv', 'row': 0, 'Date received': '03/27/25', 'Product': 'Student loan', 'Sub-product': 'Federal student loan servicing', 'Issue': 'Dealing with your lender or servicer', 'Sub-issue': 'Trouble with how payments are being handled', 'Consumer complaint narrative': "The federal student loan COVID-19 forbearance program ended in XX/XX/XXXX. However, payments were not re-amortized on my federal student loans currently serviced by Nelnet until very recently. The new payment amount that is effective starting with the XX/XX/XXXX payment will nearly double my payment from {$180.00} per month to {$360.00} per month. I'm fortunate that my current financial position allows me to be able to handle the increased payment amount, but I am sure there are likely many borrowers who are not in the same position. The re-amortization should have occurred once the forbearance ended to reduce the impact to borrowers.", 'Company public response': 'None', 'Company'

## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "LoanComplaints".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [6]:
from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Qdrant.from_documents(
    loan_complaint_data,
    embeddings,
    location=":memory:",
    collection_name="LoanComplaints"
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [7]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [8]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [9]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [10]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [11]:
naive_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided information, the most common issues with loans appear to be related to mismanagement and mishandling by loan servicers, including:\n\n- Errors in loan balances and interest calculations\n- Incorrect or fraudulent reporting of loan status or balances\n- Difficulty applying payments correctly or paying down principal\n- Unauthorized or unnotified transfer of loans between servicers\n- Discrepancies and inaccuracies in credit reporting\n- Challenges with repayment plans and loan forgiveness\n- Poor communication and lack of transparency from servicers\n- Receiving bad or inaccurate information about loans and repayment terms\n\nThese issues highlight systemic problems with loan servicing and management, leading to errors, increased debt burdens, and consumer hardship.'

In [12]:
naive_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Yes, some complaints did not get handled in a timely manner. Specifically, the complaint with Complaint ID 12709087 submitted to MOHELA on 03/28/25 was marked as "Not timely" for response, indicating it was not handled within the expected timeframe. Additionally, complaints with Complaint IDs 13160766 and 12832400, both submitted to Maximus Federal Services, Inc., were responded to as "Handled with explanation" and "Closed with explanation" within the designated timeframes, suggesting timely handling.\n\nHowever, the complaint with ID 12709087 clearly indicates it was not handled promptly. Overall, based on the provided data, at least one complaint was not addressed in a timely manner.'

In [13]:
naive_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans primarily due to a combination of factors highlighted in the complaints:\n\n1. **Lack of Clear Communication and Notification:** Many borrowers were not informed when their loan payments were to resume, when loan servicers changed, or about their repayment obligations. For example, several complaints mention not being notified about transfer of loans between servicers or the start date of repayment periods.\n\n2. **Mismanagement and Poor Handling by Servicers:** Several complaints indicate that loan servicers failed to provide adequate information, did not offer options like repayment adjustments or refinancing, and sometimes reported delinquency without proper notification. Some borrowers were unable to access or understand their account details, making it difficult to manage payments.\n\n3. **Interest Accumulation and Compounding Issues:** Borrowers expressed concern that interest continued to accrue—sometimes rapidly—while they were unable to m

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [14]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(loan_complaint_data, )

We'll construct the same chain - only changing the retriever.

In [15]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [16]:
bm25_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

"Based on the provided context, one of the most common issues with loans, specifically federal student loans, is dealing with the lender or servicer, including disputes about fees, payment application, and incorrect or confusing information about loan balances or terms. Several complaints highlight problems such as:\n\n- Disagreements about fees charged\n- Trouble with how payments are being handled and applied\n- Receiving inaccurate or bad information about loan balances, interest calculations, or school credentials\n\nOverall, a recurring theme is that consumers often experience issues with loan servicers' transparency, accuracy, and fairness in handling repayments and account information.  \n\nTherefore, the most common issue appears to be difficulties in dealing with the lender or servicer, particularly related to fees and the handling of payments or loan information."

In [17]:
bm25_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided information, it appears that several complaints were handled in a timely manner, as indicated by the responses labeled "Company response to consumer: Closed with explanation" and "Timely response?: Yes" for multiple complaints. However, there is at least one detailed complaint where the consumer reports ongoing issues and significant delays in resolution, with prolonged wait times and repeated attempts to get the issue corrected over several years. \n\nTherefore, while many complaints received timely responses, some issues, especially those involving complex disputes or corrections, seem to have experienced delays or lack of resolution over an extended period. \n\nIn summary:  \nYes, some complaints did not get handled promptly or fully resolved in a timely manner.'

In [18]:
bm25_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for various reasons, including issues with the loan servicers such as being steered into the wrong types of forbearances, lack of communication or failure to respond to requests for deferment or forbearance, technical problems like payments being reversed or not processed correctly, and sometimes being unaware of transfers to different companies or changes in their accounts. Additionally, some borrowers experienced difficulties due to mismanagement or deceptive practices by loan servicers, which led to late or missed payments, negative impacts on credit scores, and feeling of being disempowered or misinformed about their loan status.'

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

#### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

#### ✅ Answer :

✨ Example Query:

"What complaints mention Metro 2 guidelines?"

🐍 Code Blocks to Demonstrate:

1. Embeddings (Naive) Response:

In [19]:
naive_retrieval_chain.invoke({"question" : "What complaints mention Metro 2 guidelines?"})["response"].content

'The complaints that specifically mention Metro 2 guidelines include:\n\n1. The complaint against TransUnion (Complaint ID: 13515083) where the consumer alleges that TransUnion failed to apply the appropriate dispute codes, deferment/forbearance status, and discharge claims under Metro 2 reporting standards. The complaint states that TransUnion did not correctly reflect dispute codes or suppress reporting while the dispute was ongoing, which violates Metro 2 standards.\n\n2. The complaints against EdFinancial Services (Complaint ID: 13314678) and Nelnet (Complaint ID: 13329615), where the consumers describe inaccuracies and improper reporting related to federal student loans. They mention that data furnishers must ensure information is complete and accurate per Metro 2 standards, implying issues with reporting procedures that may violate those standards.\n\n3. The complaint against Aidvantage (Complaint ID: 12896336) and others involving issues of credit reporting and data accuracy, wh

2. BM25 Response:

In [20]:
bm25_retrieval_chain.invoke({"question" : "What complaints mention Metro 2 guidelines?"})["response"].content

'The complaints mention Metro 2 guidelines in the context of violations related to credit reporting procedures. Specifically, the complaints state that TransUnion failed to correctly reflect dispute codes, deferment/forbearance status, and discharge claims according to Metro 2 reporting standards. They highlight that TransUnion did not apply appropriate codes or suppress reporting while disputes were ongoing, indicating violations of Metro 2 compliance standards.'

3. Comparison Analysis:

In [21]:
# Compare which companies each method finds
bm25_docs = bm25_retriever.invoke("What complaints mention Metro 2 guidelines?")
embeddings_docs = naive_retriever.invoke("What complaints mention Metro 2 guidelines?")

print("BM25 companies found:", [doc.metadata.get('Company') for doc in bm25_docs[:5]])
print("Embeddings companies found:", [doc.metadata.get('Company') for doc in embeddings_docs[:5]])

BM25 companies found: ['TRANSUNION INTERMEDIATE HOLDINGS, INC.', 'EdFinancial Services', 'TRANSUNION INTERMEDIATE HOLDINGS, INC.', 'EdFinancial Services']
Embeddings companies found: ['TRANSUNION INTERMEDIATE HOLDINGS, INC.', 'TRANSUNION INTERMEDIATE HOLDINGS, INC.', 'EdFinancial Services', 'MOHELA', 'Maximus Federal Services, Inc.']


📊 Justification:

1. Embeddings Results: 

TRANSUNION INTERMEDIATE HOLDINGS, INC., EdFinancial Services, MOHELA, Maximus Federal Services, Inc.

2. BM25 Results: 

TRANSUNION INTERMEDIATE HOLDINGS, INC., EdFinancial Services

🎯 Why BM25 is Better:

�� Evidence from complaints.csv:

🔹 TransUnion has Metro 2 complaints (Lines 139, 3817)

🔹 EdFinancial has Metro 2 complaints (Line 770)

🔹 MOHELA has NO Metro 2 complaints

🔹 Maximus Federal Services, Inc. has NO Metro 2 complaints

BM25 is better because it found only companies with actual Metro 2 complaints(TransUnion, EdFinancial), while embeddings included companies (MOHELA, Maximus Federal Services, Inc.) that have NO Metro 2 complaints in the dataset.

🏆 Conclusion:

BM25 demonstrates superior precision by finding only documents containing the exact "Metro 2" keyword, while embeddings return semantically related but irrelevant documents.

## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [22]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [23]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [24]:
contextual_compression_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issue with loans, particularly as reflected in the provided complaints, appears to be problems related to dealing with lenders or servicers. This includes receiving bad or incorrect information about the loan, errors in loan balances, misapplied payments, wrongful denials of payment plans, and mishandling of loan data. Specific problems also include lack of communication, unauthorized transfers, and disputes over account accuracy and privacy violations.'

In [25]:
contextual_compression_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided information, there are indications that some complaints did not get handled in a timely manner. For example:\n\n- The complaint regarding the loan account review and response delay has been open since an unspecified time ("XXXX") and remains unresolved after nearly 18 months, with the complainant still awaiting resolution.\n- The issue with payments not being applied to the loan account has been ongoing for over 2-3 weeks without resolution.\n- The complaint about the main issue not being addressed and ongoing auto pay issues has been outstanding for more than 2-3 weeks.\n\nTherefore, yes, some complaints in the provided data did not get handled in a timely manner.'

In [26]:
contextual_compression_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans primarily due to a lack of clear communication, understanding, and transparency from loan servicers. Many borrowers were unaware that they would need to repay their loans and did not receive sufficient information about the terms, interest accumulation, or their repayment obligations. Some faced difficulties because loans were transferred between different servicers without notice, and they encountered inaccurate account information and unhelpful payment options like forbearance or deferment, which allowed interest to continue accumulating and increased the total debt over time. Additionally, financial hardships, stagnant wages, and the complexity of interest calculations contributed to borrowers being unable to make payments or pay off their loans effectively.'

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [27]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
)

In [28]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [29]:
multi_query_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

"Based on the provided complaints and data, the most common issues with loans appear to be:\n\n1. **Dealing with your lender or servicer** – including poor communication, errors in loan balances, misapplied payments, wrongful denials of payment plans, and difficulty in getting accurate information.\n2. **Problems with how payments are being handled** – such as inability to apply extra funds to principal, payments being applied to interest only, or inability to pay off smaller loans quickly.\n3. **Discrepancies or inaccuracies in loan balances and terms** – including incorrect account information, unexplained increases, or loans appearing under someone else's name.\n4. **Bad information or lack of proper validation of loans** – including disputes over the legitimacy of the debt, missing or incomplete documentation, and concerns about loan transfer or servicing rights.\n5. **Struggling to repay or problem with repayment plans** – such as being steered into forbearance with accumulating i

In [30]:
multi_query_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints data, yes, there are several complaints indicating that issues were not handled in a timely manner. Specifically:\n\n- One complaint from 03/28/25 (Complaint ID: 12709087) states "It has been nearly 18 months with no resolution."\n- Multiple complaints from other dates mention delays of over a year or more in receiving responses or resolutions, such as those from 04/21/25, 04/18/25, 05/06/25, and 05/02/25, where consumers report substantial delays, unresponsive customer service, or ongoing unresolved issues.\n\nAdditionally, many complaints explicitly mention that responses or actions were not timely, with some indicating wait times of several weeks or over a year before resolution.\n\nTherefore, the answer is: **Yes, some complaints did not get handled in a timely manner.**'

In [31]:
multi_query_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans primarily due to factors such as lack of proper information about repayment options, mismanagement by servicers, and financial hardships. Several complaints highlight issues like being steered into forbearance without understanding that interest would continue to accrue and compound, or being coerced into consolidations that increased their loan balances and reset forgiveness timelines. Others faced unanticipated delays or errors in loan servicing, or received inadequate assistance when seeking alternative payment plans or relief options. Additionally, circumstances like unemployment, health issues, or financial hardship made repayment difficult or impossible, leading to overdue balances and negative credit impacts.'

#### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

#### ✅ Answer :

💡 What is Recall?

Recall measures how many relevant documents are found out of all relevant documents in the database. 

Higher recall = finding more relevant documents.

🔄 How Multiple Reformulations Help:

1. Different Query Perspectives:

🔹 Original query: "loan payment issues"

🔹 Reformulations:

     "payment problems with loans"
     "loan repayment difficulties"
     "student loan payment complaints"
     "auto-debit payment failures"

2. Capturing Different Document Variations:

🔹 Some documents use "payment problems" while others use "repayment difficulties"

🔹 Different reformulations match different document phrasings

🔹 More reformulations = higher chance of finding relevant documents

3. Overcoming Vocabulary Mismatch:

🔹User might say "auto-debit" but documents say "automatic payments"

🔹Reformulations bridge this vocabulary gap

🔹Each reformulation targets different document language patterns

📊 Multi-Query Retriever Implementation:

In [42]:
from langchain.retrievers import MultiQueryRetriever

# Create multi-query retriever
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, 
    llm=chat_model
)

# Usage example
multi_query_retriever.invoke("What are loan payment issues?")

[Document(metadata={'source': './data/complaints.csv', 'row': 375, 'Date received': '04/06/25', 'Product': 'Student loan', 'Sub-product': 'Federal student loan servicing', 'Issue': 'Dealing with your lender or servicer', 'Sub-issue': 'Received bad information about your loan', 'Consumer complaint narrative': 'I am writing to formally submit a complaint regarding the services provided by MOHELA, loans also formerly known as XXXX XXXX, XXXX, and XXXX. \nI started college in XXXX at XXXX XXXX XXXX and quickly felt taken advantage of by my student loans. After graduating from XXXX, I struggled financially and called to find a way to repay my loans. I was advised to put them in forbearance, but no one explained that interest would accrue and compound to an unbelievable amount. \nI repeatedly asked for alternative solutions, but each time, I was pushed back into forbearance, with the interest growing. In XXXX, I was told to refinance my loans into XXXX larger loans, which only made things ha

�� Evidence from the Results:

🔺Single query found 10 relevant documents

🔺Multiple reformulations found 25 relevant documents

🔺Improvement: 

150% more documents found (25 vs 10) - More comprehensive coverage of the relevant document space

In [49]:
# Compare single vs multi-query retrieval
single_docs = naive_retriever.invoke("What are loan payment issues?")
multi_docs = multi_query_retriever.invoke("What are loan payment issues?")

print("Single query results:", len(single_docs))
print("Multi-query results:", len(multi_docs))

Single query results: 10
Multi-query results: 25


✨ Summary: 

Generating multiple reformulations of a user query improves recall by creating diverse search perspectives that capture different ways the same information might be expressed in documents.

This approach ensures that relevant documents are found even when they use different terminology than the original query, leading to higher overall recall by retrieving more relevant documents that might have been missed with a single query approach.

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. Each un-split "document" will be designated as a "parent document" (You could use larger chunks of document as well, but our data format allows us to consider the overall document as the parent chunk)
2. Store those "parent documents" in a memory store (not a VectorStore)
3. We will chunk each of those documents into smaller documents, and associate them with their respective parents, and store those in a VectorStore. We'll call those "child chunks".
4. When we query our Retriever, we will do a similarity search comparing our query vector to the "child chunks".
5. Instead of returning the "child chunks", we'll return their associated "parent chunks".

Okay, maybe that was a few steps - but the basic idea is this:

- Search for small documents
- Return big documents

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information that is encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by creating our "parent documents" and defining a `RecursiveCharacterTextSplitter`.

In [50]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_docs = loan_complaint_data
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [51]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="full_documents", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [52]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [53]:
parent_document_retriever.add_documents(parent_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [54]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [55]:
parent_document_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issue with loans, based on the provided complaints, appears to involve problems with debt management and reporting, such as errors in loan balances, misapplied payments, wrongful denials of payment plans, and inaccurate or unverified information in credit reports. Specifically, issues like incorrect account information, unfair interest rate increases, misreported balances, and problems due to loan servicing errors are prominent.'

In [56]:
parent_document_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided context, multiple complaints were marked as "No" for being handled in a timely manner. Specifically, the complaints regarding delays in responses and unresolved issues with student loan servicing providers by MOHELA and Aidvantage indicate that these complaints did not receive responses within a timely timeframe.'

In [57]:
parent_document_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for various reasons, including financial hardship, lack of proper information or communication from loan servicers, and issues related to the stability and transparency of the institutions involved. For example, some individuals experienced severe financial hardship after graduation, making it difficult to make payments, especially when their educational institution faced problems or misrepresented the value of their degree. Others faced difficulties due to poor communication from loan servicers, such as failure to notify them of repayment obligations or changes in loan ownership, which hindered their ability to make timely payments. Additionally, issues like the collapse or mismanagement of educational institutions and deceptive practices by third-party companies contributed to the inability to repay loans.'

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [58]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [59]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [60]:
ensemble_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issue with loans, based on the complaints data provided, appears to be dealing with the handling of student loans by lenders or servicers, including problems such as:\n\n- Errors in loan balances and account information\n- Misapplied payments and failure to properly update payment status\n- Wrongful denials of repayment plans or forgiveness and issues with how payments are being managed\n- Receiving bad or inconsistent information about loan terms, balances, and interest\n- Problems with loan transfers, improper servicing, and transfer of loans without proper notification\n- Issues related to loan consolidation, with lack of transparency and improper handling\n- Errors in credit reporting and inaccurate reflection of the loan status affecting credit scores\n\nAmong these, a recurring theme is the mishandling of loan accounts—such as errors in balances, misapplication of payments, or inadequate communication—which frequently impact borrower credit and financial planning

In [61]:
ensemble_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, yes, there are several instances indicating complaints that did not get handled in a timely manner. For example:\n\n- A complaint received on 03/28/25 (Complaint ID: 12709087) notes that the response was "No" for timeliness and indicates that the complaint was not addressed promptly.\n- Multiple complaints (e.g., Complaint IDs 12935889, 12950199, 13056764) acknowledge delays or lack of responses, with some complaints specifically mentioning waiting over extended periods, such as several hours or more than a year without resolution.\n- One complaint (ID: 12935889) was marked as "No" for timely response, and others state that the issue remains unresolved despite multiple follow-ups.\n\nThese patterns suggest that some complaints did indeed not get handled in a timely manner.'

In [62]:
ensemble_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans primarily because of a combination of factors such as lack of clear communication from lenders and servicers, mismanagement, misconceptions about repayment options, and financial hardships. Many borrowers were not properly informed about their repayment obligations, eligibility for income-driven repayment plans, or the impact of forbearance and deferment on their debt, especially the accumulation of interest. Additionally, some experienced difficulties due to changes in loan servicers without proper notification, incorrect or outdated contact information, and unpredictable or opaque loan handling practices. These systemic issues made it challenging for borrowers to manage their loans effectively, leading to delays, default, or default-like reporting, which adversely affected their credit and financial stability.\n'

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [63]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [64]:
semantic_documents = semantic_chunker.split_documents(loan_complaint_data[:20])

Let's create a new vector store.

In [65]:
semantic_vectorstore = Qdrant.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="Loan_Complaint_Data_Semantic_Chunks"
)

We'll use naive retrieval for this example.

In [66]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [67]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [68]:
semantic_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided complaints data, the most common issue with loans appears to be problems related to how payments are being handled and communication issues with lenders or servicers. Specific frequent issues include:\n\n- Trouble with how payments are being processed, auto-debit issues, and discrepancies in payment amounts.\n- Lack of clear communication or transparency from the loan servicers.\n- Errors in reporting account statuses, such as incorrect default or delinquency notices.\n- Difficulties in reconciling account information, account changes, or loan balances.\n- Problems arising from updating or re-amortizing payment plans after forbearance or end of COVID-19 relief programs.\n- Issues with reporting and privacy violations, including unauthorized data breaches.\n\nIn summary, the most common issue is problems related to payment processing, communication, and account reporting errors with loan servicers.'

In [69]:
semantic_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, several complaints indicate that complaints were not handled in a timely manner. For example:\n\n- Complaint ID 13331376 from Nelnet, Inc. (IN) on 05/04/25 states that despite acknowledging receipt of complaints, Nelnet never responded to the certified mail, nor provided answers to the raised questions, implying a lack of timely handling despite the company\'s response being "Closed with explanation."\n- Similarly, other complaints mention issues such as delayed responses, unresolved disputes, or ongoing problems despite the complaints being submitted.\n\nTherefore, yes, some complaints did not get handled in a timely manner.'

In [70]:
semantic_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

"People failed to pay back their loans for various reasons, including administrative issues, miscommunication, technical errors, and alleged misconduct by loan servicers. For example, some borrowers experienced problems with re-amortization of their payments after the end of the COVID-19 forbearance, leading to unexpectedly higher payments. Others faced difficulties due to lack of transparency, bad information from lenders or servicers, and delays or errors in processing payments. Additionally, some borrowers encountered disputes over the legitimacy or status of their loans, such as claims of defaults or illegal reporting, which hindered their ability or willingness to continue repayment. Overall, these issues highlight how administrative failures, mismanagement, and inadequate communication can contribute to borrowers' inability to repay their loans."

#### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

####  ✅ Answer :

How Semantic Chunking Behaves with Short, Repetitive Sentences:

When sentences are short and highly repetitive (like FAQs), semantic chunking behaves poorly because:

1. Embedding Similarity Issues:

🔹 Short sentences contain minimal semantic variation

🔹 Repetitive phrases create nearly identical embeddings

🔹 The algorithm struggles to differentiate between similar chunks

2. Poor Search Precision:

🔹 Multiple chunks with similar content produce similar vector representations

🔹 Search results become indistinguishable and unhelpful

🔹 Users get redundant, low-quality matches

3. Context Loss:

🔹 Short chunks fragment related information

🔹 Important context gets split across multiple chunks

🔹 Complete meaning becomes difficult to reconstruct

4. Retrieval Degradation:

🔹 The semantic search becomes less effective

🔹 Results lack diversity and relevance

🔹 The chunking strategy fails to serve its purpose

🔄 How to Adjust the Algorithm:

📏 1. Increase Chunk Size:

In [ ]:
# Instead of small chunks
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

# Use larger chunks for repetitive content
child_splitter = RecursiveCharacterTextSplitter(chunk_size=1500)

🎯 2. Use Semantic Splitting:

In [ ]:
from langchain_text_splitters import SemanticChunkSplitter

# Split based on meaning, not just characters
semantic_splitter = SemanticChunkSplitter(
    embeddings=embeddings,
    breakpoint_threshold_type="percentile"
)

📋 3. Implement Overlap Strategy:

In [ ]:
# Add overlap between chunks to preserve context
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

🏷️ 4. Use Metadata-Based Grouping:

🔹 # Group related FAQ items together

🔹 # Instead of splitting each FAQ separately

🔹 # Keep related questions in the same chunk

📊 5. Adjust Retrieval Strategy:

🔹 # Use different retrieval methods for repetitive content

🔹 # Consider BM25 for exact keyword matching

🔹 # Or use hybrid approaches combining semantic + keyword search

💡 6. Implement Deduplication:

🔹 # Remove or merge very similar chunks

🔹 # Prevent redundant results

🔹 # Focus on unique content

🎯 Benefits of Algorithm Adjustments:

🔸 Better Context: Larger chunks preserve meaning

🔸 Reduced Redundancy: Deduplication eliminates repetition

🔸 Improved Retrieval: More meaningful search results

🔸 Enhanced Relevance: Better matching for repetitive content

Example:

1. Before (small chunks):

FAQ1: "How do I reset my password?"

FAQ2: "How do I change my password?"

FAQ3: "How do I update my password?"

2. After (larger chunks):

Chunk: "Password management: reset, change, and update procedures with step-by-step instructions"

Result: More meaningful, less repetitive, better search results.

✨ Summary:

With short, repetitive sentences like FAQs, semantic chunking behaves poorly because the minimal semantic variation creates nearly identical embeddings, making it difficult to differentiate between similar chunks and leading to poor search precision with redundant, low-quality matches.
 
To adjust the algorithm, you should increase chunk size to preserve more context, use semantic splitting based on meaning rather than character count, implement overlap strategies between chunks, group related FAQ items together in metadata-based grouping, and consider using different retrieval methods like BM25 for exact keyword matching instead of relying solely on semantic similarity.


# 🤝 Breakout Room Part #2

#### 🏗️ Activity #1

Your task is to evaluate the various Retriever methods against eachother.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparision between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

####  ✅ Answer :

#### Creation of Golden Dataset :

In [71]:
# BREAKOUT ROOM PART #2: Step 1 - Golden Dataset Creation
# ========================================================

print("📋 Step 1: Creating Golden Dataset...")

# Setup LLM and embeddings (exactly as in working notebook)
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

# Create synthetic dataset with 5 test cases (as requested)
generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
ragas_dataset = generator.generate_with_langchain_docs(loan_complaint_data[:20], testset_size=5)

print(f"🎯 Created synthetic dataset with {len(ragas_dataset)} test cases")
print("✅ Golden dataset created successfully!")

# Display golden dataset (exactly as in working notebook)
ragas_dataset.to_pandas()


📋 Step 1: Creating Golden Dataset...


Applying SummaryExtractor:   0%|          | 0/14 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/20 [00:00<?, ?it/s]

Node ba7d3c1a-21a5-4adb-9edd-aef0ff7f4d70 does not have a summary. Skipping filtering.
Node 627fb807-53f8-4295-a1ce-e23b7ff59f8f does not have a summary. Skipping filtering.
Node 6cdf822e-8db9-43ac-a051-3ff039d3556a does not have a summary. Skipping filtering.
Node 7c3a0a8b-0c84-481f-a450-82f984e35041 does not have a summary. Skipping filtering.
Node a2579928-2f31-4c69-a8d1-827b416c3e74 does not have a summary. Skipping filtering.
Node 9b4432d8-03f4-46df-b1fb-165a7d0bc61f does not have a summary. Skipping filtering.


Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/54 [00:00<?, ?it/s]

...

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/6 [00:00<?, ?it/s]

🎯 Created synthetic dataset with 6 test cases
✅ Golden dataset created successfully!


,user_input,reference_contexts,reference,synthesizer_name
0,Does Nelnet re-amortize federal student loans ...,[The federal student loan COVID-19 forbearance...,The context indicates that payments were not r...,single_hop_specifc_query_synthesizer
1,What is Aidvantage's role in student loan repa...,[I submitted my annual Income-Driven Repayment...,Aidvantage is involved in managing student loa...,single_hop_specifc_query_synthesizer
2,How does FERPA protect student data and what h...,[My personal and financial data was compromise...,My personal and financial data was compromised...,single_hop_specifc_query_synthesizer
3,As a legal advocate for student privacy rights...,"[<1-hop>\n\nI set up autopay with AidVantage, ...",The situation with AidVantage demonstrates pot...,multi_hop_specific_query_synthesizer
4,How does the Department of Education's abolish...,[<1-hop>\n\nThis is a formal legal demand for ...,The abolishment of the Department of Education...,multi_hop_specific_query_synthesizer
5,If Nelnet is responsible for my student loan i...,[<1-hop>\n\nThis is a formal legal demand for ...,"The context shows that Nelnet, as the loan ser...",multi_hop_specific_query_synthesizer


#### LangSmith Dataset Setup :

In [73]:
# BREAKOUT ROOM PART #2: Step 2 - LangSmith Dataset Setup
# ========================================================

print("🚀 Setting up LangSmith evaluation framework...")

# Create LangSmith client (using different variable name to avoid conflict)
from langsmith import Client
langsmith_client = Client()

# Create LangSmith dataset
dataset_name = "Advanced_Retrieval_Evaluation_Dataset"

langsmith_dataset = langsmith_client.create_dataset(
    dataset_name=dataset_name,
    description="Advanced Retrieval Methods Evaluation Dataset"
)

print(f"✅ Created LangSmith dataset: {dataset_name}")

# Add golden dataset examples to LangSmith (FIXED - using ragas_dataset)
print("📝 Adding golden dataset examples to LangSmith...")

for data_row in ragas_dataset.to_pandas().iterrows():  # ✅ FIXED: using ragas_dataset
    langsmith_client.create_example(
        inputs={
            "question": data_row[1]["user_input"]
        },
        outputs={
            "answer": data_row[1]["reference"]
        },
        metadata={
            "context": data_row[1]["reference_contexts"]
        },
        dataset_id=langsmith_dataset.id
    )

print(f"✅ Added {len(ragas_dataset)} examples to LangSmith dataset")  # ✅ FIXED: using ragas_dataset

🚀 Setting up LangSmith evaluation framework...
✅ Created LangSmith dataset: Advanced_Retrieval_Evaluation_Dataset
📝 Adding golden dataset examples to LangSmith...
✅ Added 6 examples to LangSmith dataset


#### Evaluation with BOTH LangSmith + Ragas Metrics :

In [74]:
# BREAKOUT ROOM PART #2: Evaluation with BOTH LangSmith + Ragas (Separate Blocks)
# ================================================================================

print("🚀 Starting evaluation...")

# Set up evaluation LLM
eval_llm = ChatOpenAI(model="gpt-4.1-nano")

# Set up LangSmith evaluators (exact same as reference)
from langsmith.evaluation import LangChainStringEvaluator, evaluate

qa_evaluator = LangChainStringEvaluator("qa", config={"llm": eval_llm})

labeled_helpfulness_evaluator = LangChainStringEvaluator(
    "labeled_criteria",
    config={
        "criteria": {
            "helpfulness": (
                "Is this submission helpful to the user,"
                " taking into account the correct reference answer?"
            )
        },
        "llm": eval_llm
    },
    prepare_data=lambda run, example: {
        "prediction": run.outputs["output"],
        "reference": example.outputs["answer"],
        "input": example.inputs["question"],
    }
)

# Set up Ragas evaluator and metrics
evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))

# Ragas metrics
ragas_metrics = [
    ContextPrecision(llm=evaluator_llm),
    LLMContextRecall(),
    Faithfulness(), 
    FactualCorrectness(),
    ResponseRelevancy(),
    ContextEntityRecall(),
    NoiseSensitivity()
]

# Configure evaluation with timeout
custom_run_config = RunConfig(timeout=180)

# Create evaluation chains for each retriever
def create_evaluation_chain(retriever, name):
    return (
        {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
        | rag_prompt | chat_model | StrOutputParser()
    )

print("✅ Setup complete! Ready for individual retriever evaluations.")

🚀 Starting evaluation...
✅ Setup complete! Ready for individual retriever evaluations.


#### Naive Retriever Evaluation :

In [75]:
# BREAKOUT ROOM PART #2: Naive Retriever Evaluation (Fixed with Results)
# =====================================================================

print("🔄 Evaluating naive retriever...")

eval_chain = create_evaluation_chain(naive_retriever, "naive")

# Run LangSmith evaluation (generates links)
from langsmith.evaluation import evaluate as langsmith_evaluate

langsmith_result = langsmith_evaluate(
    eval_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator
    ],
    metadata={"retriever_type": "naive"},
)

# Process dataset for Ragas evaluation (WORKING - using correct Ragas dataset)
for test_row in ragas_dataset:
    response = eval_chain.invoke({"question": test_row.eval_sample.user_input})
    test_row.eval_sample.response = response
    test_row.eval_sample.retrieved_contexts = [doc.page_content for doc in naive_retriever.invoke(test_row.eval_sample.user_input)]

# Convert to EvaluationDataset
evaluation_dataset = EvaluationDataset.from_pandas(ragas_dataset.to_pandas())

# Run Ragas evaluation (FIXED - increased timeout and display results)
from ragas import evaluate as ragas_evaluate

# Increase timeout to avoid timeouts
custom_run_config = RunConfig(timeout=300)  # 5 minutes

ragas_result = ragas_evaluate(
    dataset=evaluation_dataset,
    metrics=ragas_metrics,
    llm=evaluator_llm,
    run_config=custom_run_config
)

# Display Ragas results
print("📊 Ragas Evaluation Results for Naive Retriever:")
print(ragas_result)

print("✅ naive retriever evaluation complete")

🔄 Evaluating naive retriever...
View the evaluation results for experiment: 'only-tendency-47' at:
https://smith.langchain.com/o/9cc120d2-ca66-4b3b-9545-ca0497b68deb/datasets/18cc962c-69c3-4880-a423-bca95db2063e/compare?selectedSessions=c060cdf5-69c0-40de-9a71-047eb65574bb




0it [00:00, ?it/s]

Evaluating:   0%|          | 0/42 [00:00<?, ?it/s]

Exception raised in Job[5]: TimeoutError()
Exception raised in Job[12]: TimeoutError()
Exception raised in Job[26]: TimeoutError()
Exception raised in Job[27]: TimeoutError()
Exception raised in Job[34]: TimeoutError()
Exception raised in Job[41]: TimeoutError()


📊 Ragas Evaluation Results for Naive Retriever:
{'context_precision': 0.7492, 'context_recall': 0.9167, 'faithfulness': 0.9444, 'factual_correctness': 0.8700, 'answer_relevancy': 0.6380, 'context_entity_recall': 0.6250, 'noise_sensitivity_relevant': 0.0000}
✅ naive retriever evaluation complete


#### BM25 Retriever Evaluation :

In [76]:
# BREAKOUT ROOM PART #2: BM25 Retriever Evaluation (Fixed with Results)
# =====================================================================

print("🔄 Evaluating BM25 retriever...")

eval_chain = create_evaluation_chain(bm25_retriever, "bm25")

# Run LangSmith evaluation (generates links)
from langsmith.evaluation import evaluate as langsmith_evaluate

langsmith_result = langsmith_evaluate(
    eval_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator
    ],
    metadata={"retriever_type": "bm25"},
)

# Process dataset for Ragas evaluation (WORKING - using correct Ragas dataset)
for test_row in ragas_dataset:
    response = eval_chain.invoke({"question": test_row.eval_sample.user_input})
    test_row.eval_sample.response = response
    test_row.eval_sample.retrieved_contexts = [doc.page_content for doc in bm25_retriever.invoke(test_row.eval_sample.user_input)]

# Convert to EvaluationDataset
evaluation_dataset = EvaluationDataset.from_pandas(ragas_dataset.to_pandas())

# Run Ragas evaluation (FIXED - increased timeout and display results)
from ragas import evaluate as ragas_evaluate

# Increase timeout to avoid timeouts
custom_run_config = RunConfig(timeout=300)  # 5 minutes

ragas_result = ragas_evaluate(
    dataset=evaluation_dataset,
    metrics=ragas_metrics,
    llm=evaluator_llm,
    run_config=custom_run_config
)

# Display Ragas results
print("📊 Ragas Evaluation Results for BM25 Retriever:")
print(ragas_result)

print("✅ BM25 retriever evaluation complete")

🔄 Evaluating BM25 retriever...
View the evaluation results for experiment: 'terrific-heart-51' at:
https://smith.langchain.com/o/9cc120d2-ca66-4b3b-9545-ca0497b68deb/datasets/18cc962c-69c3-4880-a423-bca95db2063e/compare?selectedSessions=3b439a46-c561-41a3-bd01-231a449413eb




0it [00:00, ?it/s]

Evaluating:   0%|          | 0/42 [00:00<?, ?it/s]

Exception raised in Job[27]: ValueError(setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (4,) + inhomogeneous part.)
Exception raised in Job[33]: TimeoutError()


📊 Ragas Evaluation Results for BM25 Retriever:
{'context_precision': 0.8333, 'context_recall': 0.8611, 'faithfulness': 1.0000, 'factual_correctness': 0.8717, 'answer_relevancy': 0.9499, 'context_entity_recall': 0.3804, 'noise_sensitivity_relevant': 0.0200}
✅ BM25 retriever evaluation complete


#### Contextual Compression(Cohere Reranking) Retriever Evaluation:

In [77]:
# BREAKOUT ROOM PART #2: Contextual Compression (Cohere Reranking) Retriever Evaluation
# =====================================================================================

import time

print("�� Evaluating Contextual Compression (Cohere Reranking) retriever...")

# Add a delay between evaluations to avoid rate limits
time.sleep(60)  # Wait 1 minute before retrying

eval_chain = create_evaluation_chain(compression_retriever, "contextual_compression")

# Run LangSmith evaluation (generates links)
from langsmith.evaluation import evaluate as langsmith_evaluate

langsmith_result = langsmith_evaluate(
    eval_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator
    ],
    metadata={"retriever_type": "contextual_compression_cohere_reranking"},
)

# Process dataset for Ragas evaluation (FIXED - with rate limiting)
for i, test_row in enumerate(ragas_dataset):
    try:
        response = eval_chain.invoke({"question": test_row.eval_sample.user_input})
        test_row.eval_sample.response = response
        test_row.eval_sample.retrieved_contexts = [doc.page_content for doc in compression_retriever.invoke(test_row.eval_sample.user_input)]
        
        # Add delay between calls to avoid rate limits (10 calls/minute = 6 seconds between calls)
        if i < len(ragas_dataset) - 1:  # Don't delay after the last call
            time.sleep(6)  # Wait 6 seconds between each call
            
    except Exception as e:
        print(f"⚠️ Error processing test case {i+1}: {e}")
        # If rate limited, wait longer and retry
        if "TooManyRequestsError" in str(e):
            print("🔄 Rate limited, waiting 60 seconds...")
            time.sleep(60)
            continue
        continue

# Convert to EvaluationDataset
evaluation_dataset = EvaluationDataset.from_pandas(ragas_dataset.to_pandas())

# Run Ragas evaluation (FIXED - increased timeout and display results)
from ragas import evaluate as ragas_evaluate

# Increase timeout to avoid timeouts
custom_run_config = RunConfig(timeout=300)  # 5 minutes

ragas_result = ragas_evaluate(
    dataset=evaluation_dataset,
    metrics=ragas_metrics,
    llm=evaluator_llm,
    run_config=custom_run_config
)

# Display Ragas results
print("📊 Ragas Evaluation Results for Contextual Compression (Cohere Reranking) Retriever:")
print(ragas_result)

print("✅ Contextual Compression (Cohere Reranking) retriever evaluation complete")

�� Evaluating Contextual Compression (Cohere Reranking) retriever...
View the evaluation results for experiment: 'memorable-manager-39' at:
https://smith.langchain.com/o/9cc120d2-ca66-4b3b-9545-ca0497b68deb/datasets/18cc962c-69c3-4880-a423-bca95db2063e/compare?selectedSessions=6bec91a2-8e49-46c6-afde-935cf60ae501




0it [00:00, ?it/s]

⚠️ Error processing test case 6: status_code: 429, body: data=None id='a009c845-726a-43fc-bf1e-5320bd326d28' message="You are using a Trial key, which is limited to 10 API calls / minute. You can continue to use the Trial key for free or upgrade to a Production key with higher rate limits at 'https://dashboard.cohere.com/api-keys'. Contact us on 'https://discord.gg/XW44jPfYJu' or email us at support@cohere.com with any questions"


Evaluating:   0%|          | 0/42 [00:00<?, ?it/s]

📊 Ragas Evaluation Results for Contextual Compression (Cohere Reranking) Retriever:
{'context_precision': 1.0000, 'context_recall': 0.9167, 'faithfulness': 0.9841, 'factual_correctness': 0.8683, 'answer_relevancy': 0.7843, 'context_entity_recall': 0.4286, 'noise_sensitivity_relevant': 0.0551}
✅ Contextual Compression (Cohere Reranking) retriever evaluation complete


#### Multi-Query Retriever Evaluation :

In [78]:
# BREAKOUT ROOM PART #2: Multi-Query Retriever Evaluation
# ========================================================

print("🔄 Evaluating Multi-Query retriever...")

eval_chain = create_evaluation_chain(multi_query_retriever, "multi_query")

# Run LangSmith evaluation (generates links)
from langsmith.evaluation import evaluate as langsmith_evaluate

langsmith_result = langsmith_evaluate(
    eval_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator
    ],
    metadata={"retriever_type": "multi_query"},
)

# Process dataset for Ragas evaluation (WORKING - using correct Ragas dataset)
for test_row in ragas_dataset:
    response = eval_chain.invoke({"question": test_row.eval_sample.user_input})
    test_row.eval_sample.response = response
    test_row.eval_sample.retrieved_contexts = [doc.page_content for doc in multi_query_retriever.invoke(test_row.eval_sample.user_input)]

# Convert to EvaluationDataset
evaluation_dataset = EvaluationDataset.from_pandas(ragas_dataset.to_pandas())

# Run Ragas evaluation (FIXED - increased timeout and display results)
from ragas import evaluate as ragas_evaluate

# Increase timeout to avoid timeouts
custom_run_config = RunConfig(timeout=300)  # 5 minutes

ragas_result = ragas_evaluate(
    dataset=evaluation_dataset,
    metrics=ragas_metrics,
    llm=evaluator_llm,
    run_config=custom_run_config
)

# Display Ragas results
print("📊 Ragas Evaluation Results for Multi-Query Retriever:")
print(ragas_result)

print("✅ Multi-Query retriever evaluation complete")

🔄 Evaluating Multi-Query retriever...
View the evaluation results for experiment: 'clear-value-10' at:
https://smith.langchain.com/o/9cc120d2-ca66-4b3b-9545-ca0497b68deb/datasets/18cc962c-69c3-4880-a423-bca95db2063e/compare?selectedSessions=a9e1356d-8b40-45a8-8860-6a4365b22279




0it [00:00, ?it/s]

Evaluating:   0%|          | 0/42 [00:00<?, ?it/s]

Exception raised in Job[34]: AttributeError('StringIO' object has no attribute 'sentences')
Exception raised in Job[12]: TimeoutError()
Exception raised in Job[26]: TimeoutError()
Exception raised in Job[27]: TimeoutError()
Exception raised in Job[33]: TimeoutError()
Exception raised in Job[40]: TimeoutError()
Exception raised in Job[41]: TimeoutError()


📊 Ragas Evaluation Results for Multi-Query Retriever:
{'context_precision': 0.7722, 'context_recall': 0.9167, 'faithfulness': 0.9889, 'factual_correctness': 0.8650, 'answer_relevancy': 0.6335, 'context_entity_recall': 0.3333, 'noise_sensitivity_relevant': 0.1538}
✅ Multi-Query retriever evaluation complete


#### Parent Document Retriever Evaluation :

In [80]:
# BREAKOUT ROOM PART #2: Parent Document Retriever Evaluation
# ===========================================================

print("🔄 Evaluating Parent Document retriever...")

eval_chain = create_evaluation_chain(parent_document_retriever, "parent_document")

# Run LangSmith evaluation (generates links)
from langsmith.evaluation import evaluate as langsmith_evaluate

langsmith_result = langsmith_evaluate(
    eval_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator
    ],
    metadata={"retriever_type": "parent_document"},
)

# Process dataset for Ragas evaluation (WORKING - using correct Ragas dataset)
for test_row in ragas_dataset:
    response = eval_chain.invoke({"question": test_row.eval_sample.user_input})
    test_row.eval_sample.response = response
    test_row.eval_sample.retrieved_contexts = [doc.page_content for doc in parent_document_retriever.invoke(test_row.eval_sample.user_input)]

# Convert to EvaluationDataset
evaluation_dataset = EvaluationDataset.from_pandas(ragas_dataset.to_pandas())

# Run Ragas evaluation (FIXED - increased timeout and display results)
from ragas import evaluate as ragas_evaluate

# Increase timeout to avoid timeouts
custom_run_config = RunConfig(timeout=300)  # 5 minutes

ragas_result = ragas_evaluate(
    dataset=evaluation_dataset,
    metrics=ragas_metrics,
    llm=evaluator_llm,
    run_config=custom_run_config
)

# Display Ragas results
print("📊 Ragas Evaluation Results for Parent Document Retriever:")
print(ragas_result)

print("✅ Parent Document retriever evaluation complete")

🔄 Evaluating Parent Document retriever...
View the evaluation results for experiment: 'clear-news-70' at:
https://smith.langchain.com/o/9cc120d2-ca66-4b3b-9545-ca0497b68deb/datasets/18cc962c-69c3-4880-a423-bca95db2063e/compare?selectedSessions=847c41ec-b15a-4633-864f-34ae59cbd3be




0it [00:00, ?it/s]

Evaluating:   0%|          | 0/42 [00:00<?, ?it/s]

Exception raised in Job[34]: ValueError(operands could not be broadcast together with shapes (20,) (1,22) )
Exception raised in Job[5]: TimeoutError()
Exception raised in Job[26]: TimeoutError()


📊 Ragas Evaluation Results for Parent Document Retriever:
{'context_precision': 0.7778, 'context_recall': 0.9167, 'faithfulness': 0.9946, 'factual_correctness': 0.8767, 'answer_relevancy': 0.8008, 'context_entity_recall': 0.4500, 'noise_sensitivity_relevant': 0.0000}
✅ Parent Document retriever evaluation complete


#### Ensemble Retriever Evaluation :

In [81]:
# BREAKOUT ROOM PART #2: Ensemble Retriever Evaluation (Fixed with Rate Limiting)
# ==============================================================================

import time

print("🔄 Evaluating Ensemble retriever...")

# Add a delay between evaluations to avoid rate limits
time.sleep(60)  # Wait 1 minute before retrying

eval_chain = create_evaluation_chain(ensemble_retriever, "ensemble")

# Run LangSmith evaluation (generates links)
from langsmith.evaluation import evaluate as langsmith_evaluate

langsmith_result = langsmith_evaluate(
    eval_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator
    ],
    metadata={"retriever_type": "ensemble"},
)

# Process dataset for Ragas evaluation (FIXED - with rate limiting)
for i, test_row in enumerate(ragas_dataset):
    try:
        response = eval_chain.invoke({"question": test_row.eval_sample.user_input})
        test_row.eval_sample.response = response
        test_row.eval_sample.retrieved_contexts = [doc.page_content for doc in ensemble_retriever.invoke(test_row.eval_sample.user_input)]
        
        # Add delay between calls to avoid rate limits (10 calls/minute = 6 seconds between calls)
        if i < len(ragas_dataset) - 1:  # Don't delay after the last call
            time.sleep(6)  # Wait 6 seconds between each call
            
    except Exception as e:
        print(f"⚠️ Error processing test case {i+1}: {e}")
        # If rate limited, wait longer and retry
        if "TooManyRequestsError" in str(e):
            print("🔄 Rate limited, waiting 60 seconds...")
            time.sleep(60)
            continue
        continue

# Convert to EvaluationDataset
evaluation_dataset = EvaluationDataset.from_pandas(ragas_dataset.to_pandas())

# Run Ragas evaluation (FIXED - increased timeout and display results)
from ragas import evaluate as ragas_evaluate

# Increase timeout to avoid timeouts
custom_run_config = RunConfig(timeout=300)  # 5 minutes

ragas_result = ragas_evaluate(
    dataset=evaluation_dataset,
    metrics=ragas_metrics,
    llm=evaluator_llm,
    run_config=custom_run_config
)

# Display Ragas results
print("📊 Ragas Evaluation Results for Ensemble Retriever:")
print(ragas_result)

print("✅ Ensemble retriever evaluation complete")

🔄 Evaluating Ensemble retriever...
View the evaluation results for experiment: 'fixed-chart-59' at:
https://smith.langchain.com/o/9cc120d2-ca66-4b3b-9545-ca0497b68deb/datasets/18cc962c-69c3-4880-a423-bca95db2063e/compare?selectedSessions=143da047-df45-434a-ac68-9fd8b0a4ec2d




0it [00:00, ?it/s]

Evaluating:   0%|          | 0/42 [00:00<?, ?it/s]

Exception raised in Job[6]: TimeoutError()
Exception raised in Job[12]: TimeoutError()
Exception raised in Job[13]: TimeoutError()
Exception raised in Job[20]: TimeoutError()
Exception raised in Job[26]: TimeoutError()
Exception raised in Job[27]: TimeoutError()
Exception raised in Job[33]: TimeoutError()
Exception raised in Job[34]: TimeoutError()
Exception raised in Job[40]: TimeoutError()
Exception raised in Job[41]: TimeoutError()


📊 Ragas Evaluation Results for Ensemble Retriever:
{'context_precision': 0.7686, 'context_recall': 0.9167, 'faithfulness': 1.0000, 'factual_correctness': 0.8783, 'answer_relevancy': 0.9522, 'context_entity_recall': 0.4167, 'noise_sensitivity_relevant': nan}
✅ Ensemble retriever evaluation complete


#### 📊 Advanced Retrieval Methods Evaluation Results Compilation and Analysis

 ## =======================================================

## Performance Metrics

| Retriever | Correctness | Helpfulness | Context Precision | Context Recall | Faithfulness | Factual Correctness | Answer Relevancy | Context Entity Recall | Noise Sensitivity |
|-----------|-------------|-------------|-------------------|----------------|--------------|-------------------|------------------|---------------------|-------------------|
| Naive | 0.50 | 0.50 | 0.7492 | 0.9167 | 0.9444 | 0.8700 | 0.6380 | 0.6250 | 0.0000 |
| BM25 | 0.67 | 0.40 | 0.8333 | 0.8611 | 1.0000 | 0.8717 | 0.9499 | 0.3804 | 0.0200 |
| Contextual Compression (Cohere Reranking) | 0.17 | 0.60 | 1.0000 | 0.9167 | 0.9841 | 0.8683 | 0.7843 | 0.4286 | 0.0551 |
| Multi-Query | 0.67 | 0.67 | 0.7722 | 0.9167 | 0.9889 | 0.8650 | 0.6335 | 0.3333 | 0.1538 |
| Parent Document | 0.50 | 0.60 | 0.7778 | 0.9167 | 0.9946 | 0.8767 | 0.8008 | 0.4500 | 0.0000 |
| Ensemble | 0.50 | 0.50 | 0.7686 | 0.9167 | 1.0000 | 0.8783 | 0.9522 | 0.4167 | N/A |

## Cost & Latency Metrics

| Retriever | Latency (P50) | Total Tokens | Total Cost |
|-----------|---------------|--------------|------------|
| Naive | 5.058s | 40,157 | $0.0045 |
| BM25 | 3.228s | 27,421 | $0.0033 |
| Contextual Compression (Cohere Reranking) | 3.283s | 13,637 | $0.0018 |
| Multi-Query | 7.082s | 54,810 | $0.0062 |
| Parent Document | 4.392s | 20,097 | $0.0025 |
| Ensemble | 7.664s | 88,740 | $0.0096 |

## Analysis and Recommendations

### Top Performers by Category:

Performance Metrics:

🔸 Best Correctness: BM25 and Multi-Query (67%)

🔸 Best Helpfulness: Multi-Query (67%)

🔸 Best Context Precision: Contextual Compression (100%)

🔸 Best Context Recall: Naive, Contextual Compression, Multi-Query, Parent Document, and Ensemble (91.67%)

🔸 Best Faithfulness: BM25 and Ensemble (100%)

🔸 Best Factual Correctness: Ensemble (87.83%)

🔸 Best Answer Relevancy: Ensemble (95.22%)

🔸 Best Context Entity Recall: Naive (62.50%)

🔸 Best Noise Sensitivity: Naive and Parent Document (0.0%)

Cost & Latency Metrics:

🔹 Lowest Cost: Contextual Compression ($0.0018)

🔹 Most Token Efficient: Contextual Compression (13,637 tokens)

🔹 Fastest Response: BM25 (3.228s median)

🔹 Slowest Response: Ensemble (7.664s median)

### Overall Recommendation:

BM25 is recommended as the primary retriever for this loan complaints dataset because:

1. Superior Performance: Perfect faithfulness (100%) and excellent answer relevancy (94.99%)
2. Best Correctness: 67% correctness (tied with Multi-Query for highest)
3. Fastest Response: 3.228s median latency
4. Excellent Precision: 83.33% context precision
5. Cost Effective: Only $0.0033 cost with excellent performance
6. Domain Suitability: Perfect for structured, keyword-heavy loan complaints data

### Alternative Recommendations:

🔸 For Budget-Conscious Applications: Contextual Compression ($0.0018 cost, 17% correctness but 100% context precision)

🔸 For Maximum Quality: Ensemble (87.83% factual correctness, 95.22% answer relevancy, but higher cost)

🔸 For Balanced Performance: Multi-Query (67% correctness and helpfulness, good all-around metrics)

🔸 For Entity Recognition: Naive (62.50% context entity recall, 0% noise sensitivity)

### Key Insights:

1. BM25 Dominates for This Domain: Traditional keyword-based retrieval outperforms advanced methods for loan complaints data
2. Contextual Compression Trade-off: Achieves perfect context precision (100%) but suffers in correctness (17%), indicating over-filtering
3. Cost vs. Performance Trade-off: More complex methods show higher costs without proportional performance gains
4. Domain Specificity: Loan complaints data benefits from exact keyword matching over semantic approaches
5. Token Efficiency: Contextual Compression provides the best token-to-performance ratio
6. Noise Sensitivity: Naive and Parent Document methods show perfect noise handling (0% sensitivity)

## In Essence :

BM25 is the optimal choice for this loan complaints dataset because it perfectly matches the data's characteristics. Loan complaints contain specific terminology, company names, product types, and standardized vocabulary that benefit from exact keyword matching. BM25 achieves the best balance of performance (67% correctness, 94.99% answer relevancy) and efficiency ($0.0033 cost, 3.228s latency) by leveraging the structured nature of consumer complaints. While Ensemble achieves slightly higher quality metrics (95.22% answer relevancy, 87.83% factual correctness), the performance difference is minimal compared to the significant cost and speed advantages of BM25. The dataset's consistent terminology and domain-specific language patterns make traditional keyword-based retrieval more effective than semantic approaches, making BM25 the most practical choice for this specific domain.